In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader 
import seaborn as sns

In [2]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks.early_stopping import EarlyStopping

C:\Users\praveenchakra.bh\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
cars_file = 'https://gist.githubusercontent.com/noamross/e5d3e859aa0c794be10b/raw/b999fb4425b54c63cab088c0ce2c0d6ce961a563/cars.csv'
cars = pd.read_csv(cars_file)
cars.head()

,Unnamed: 0,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [4]:
X_list = cars.wt.values
X_np = np.array(X_list, dtype=np.float32).reshape(-1,1)

In [5]:
y_list = cars.mpg.values
y_np = np.array(y_list, dtype=np.float32).reshape(-1,1)

In [6]:
X = torch.from_numpy(X_np)
y_true = torch.from_numpy(y_np)

In [7]:
class LinearRegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [8]:
train_loader = DataLoader(dataset = LinearRegressionDataset(X_np, y_np), batch_size=2)

In [9]:
class LitLinearRegression(pl.LightningModule):
    def __init__(self, input_size, output_size):
        super(LitLinearRegression, self).__init__()
        self.linear = nn.Linear(input_size, output_size)
        self.loss_fun = nn.MSELoss()
    
    def forward(self, x):
        return self.linear(x)

    def configure_optimizers(self):
        learning_rate = 0.02
        optimizer = torch.optim.SGD(self.parameters(), lr=learning_rate)
        return optimizer
    
    def training_step(self, train_batch, batch_idx):
        X, y = train_batch

        # forward pass
        y_pred = self.forward(X)

        # compute loss
        loss = self.loss_fun(y_pred, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss
    
    def val_step(self, val_batch, batch_idx):
        X, y = val_batch

        # forward pass
        y_pred = model(X)

        # compute loss
        loss = self.loss_fun(y_pred, y)
        self.log('val_loss', loss, prog_bar=True)
        return loss

In [10]:
model = LitLinearRegression(input_size=1, output_size=1)

In [11]:
model

LitLinearRegression(
  (linear): Linear(in_features=1, out_features=1, bias=True)
  (loss_fun): MSELoss()
)

In [12]:
early_stop_callback = EarlyStopping(monitor="train_loss", min_delta=0.00, patience=2, verbose=True, mode="min")

In [13]:
trainer = pl.Trainer(accelerator='gpu', devices=1, max_epochs=500, log_every_n_steps=2, callbacks=[early_stop_callback])
trainer.fit(model=model, train_dataloaders=train_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA RTX 500 Ada Generation Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ linear   │ Linear  │      2 │ train │     0 │
│ 1 │ loss_fun │ MSELoss │      0 │ train │     0 │
└───┴──────────┴─────────┴────────┴───────┴───────┘

Trainable params: 2                                                                                                
Non-trainable params: 0                                                                                            
Total params: 2                                                                                                    
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 2                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

C:\Users\praveenchakra.bh\AppData\Roaming\Python\Python39\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\praveenchakra.bh\AppData\Roaming\Python\Python39\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.
Metric train_loss improved. New best score: 35.745
Metric train_loss improved by 2.372 >= min_delta = 0.0. New best score: 33.373
Metric train_loss improved by 2.192 >= min_delta = 0.0. New best score: 31.180
Metric train_loss improved by 2.030 >= min_delta = 0.0. New best score: 29.150
Metric train_loss improved by 1.881 >= min_delta = 0.0. New best score: 27.269
Metric train_loss improved by 1.743 >= min_delta = 0.0. New best score: 25.526
Metric train_loss improved by 1.616 >= min_delta = 0.0. New best score: 23.910
Metric train_loss improved by 1.499 >= min_delta = 0.0. New best score: 22.411
Metric train_loss improved by 1.390 >= min_delta = 0.0. New be

In [14]:
trainer.current_epoch

286

In [16]:
model = LitLinearRegression.load_from_checkpoint("lightning_logs/version_0/checkpoints/epoch=285-step=4576.ckpt",
                                                 input_size=1,
                                                 output_size=1)
model.eval()

LitLinearRegression(
  (linear): Linear(in_features=1, out_features=1, bias=True)
  (loss_fun): MSELoss()
)

In [17]:
x = torch.tensor([[5.0]])
with torch.no_grad():
    y_pred = model(x.to('cuda'))[0]

print(y_pred.item())

8.080907821655273
